# Objectif

Construire le Data Warehouse dans PostgreSQL.  
**Pertinence dans le mémoire**

Cette étape matérialise :
- le système décisionnel ;
- l’architecture BI ;
- le modèle multidimensionnel.

## 1. Connexion PostgreSQL

- SQLAlchemy ;
- psycopg2.


In [7]:
#!pip install psycopg2


In [33]:
# IMPORTATION DES LIBRAIRIES
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy import text
from urllib.parse import quote_plus

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("Librairies chargées")

Librairies chargées


In [27]:
# PARAMÈTRES DE CONNEXION
USER = "postgres"

PASSWORD = "Js81pr@teg2D"

HOST = "localhost"

PORT = "5432"

DATABASE = "btp_sinistres_dwh"

In [35]:
# ENCODAGE DU MOT DE PASSE
PASSWORD = quote_plus(PASSWORD)

# CONNEXION POSTGRESQL
engine = create_engine(
    f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)

print("Connexion PostgreSQL initialisée")

Connexion PostgreSQL initialisée ✔️


In [37]:
# TEST CONNEXION
with engine.connect() as conn:

    result = conn.execute(
        text("SELECT version();")
    )

    for row in result:
        print(row)

('PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35226, 64-bit',)


In [39]:
# CHARGEMENT DES DONNÉES
dataset = pd.read_csv(
    "../data/processed/dataset_analytique.csv"
)

trc = pd.read_csv(
    "../data/processed/sinistre_trc_enrichi.csv"
)

print("Datasets chargés")

Datasets chargés


In [77]:
# APERÇU DES DONNÉES
display(dataset.head())
display(dataset.shape)

display(trc.head())
display(trc.shape)

,id_sinistre,annee,mois,date_accident,immatriculation,type_engin_sinistre,marque_sinistre,age_vehicule_ans_sinistre,etat_vehicule,departement_accident,axe_routier,type_accident,responsabilite,gravite,tiers_implique,assureur_sinistre,montant_declare_fcfa,franchise_fcfa,montant_indemnise_fcfa,delai_declaration_jours,delai_reglement_jours,statut_dossier,conducteur_anciennete_ans,type_sinistre,type_engin_parc,marque_parc,annee_mise_en_service,age_vehicule_ans_parc,departement_affectation,km_parcourus_estimes,etat_general,assureur_parc,cout_total_fcfa,classe_age,niveau_experience,score_delai,niveau_risque,sinistre_grave,frequence_sinistre,score_risque,tranche_cout,annee_accident,mois_accident,jour_semaine,est_weekend,categorie_km
0,AUTO-2022-001,2022,7,2022-07-17,PRT-0108-BJ,NIVELEUSE,LIEBHERR,13,MAUVAIS,OUÉMÉ,RN6 NATITINGOU-KÉROU,COLLISION FRONTALE,RESPONSABLE,MATÉRIEL GRAVE,OUI,COLINA,1745000,339000,1101000.0,7,119.0,RÉGLÉ,17,AUTO,NIVELEUSE,LIEBHERR,2011,13,ATLANTIQUE,504000,MAUVAIS,COLINA,1440000.0,ANCIEN,EXPERIMENTE,126.0,MOYEN,0,1,93.46,MOYEN,2022,7,Sunday,True,ELEVE
1,AUTO-2022-002,2022,3,2022-03-02,PRT-0068-BJ,BULLDOZER,LIEBHERR,10,MAUVAIS,ALIBORI,VOIRIE URBAINE PARAKOU,RENVERSEMENT,RESPONSABLE,MATÉRIEL LÉGER,OUI,SAAR-BÉNIN,242000,114000,0.0,5,0.0,EN COURS,19,AUTO,BULLDOZER,LIEBHERR,2014,10,OUÉMÉ,250000,MAUVAIS,SAAR-BÉNIN,114000.0,ANCIEN,EXPERIMENTE,5.0,FAIBLE,0,1,10.46,FAIBLE,2022,3,Wednesday,False,ELEVE
2,AUTO-2022-003,2022,9,2022-09-15,PRT-0318-BJ,CAMION BENNE,MAN,9,MAUVAIS,ATLANTIQUE,RN7 COTONOU-OUIDAH,ACCROCHAGE MINEUR,RESPONSABILITÉ PARTAGÉE,MATÉRIEL LÉGER,OUI,COLINA,471000,270000,0.0,5,0.0,EN EXPERTISE,10,AUTO,CAMION BENNE,MAN,2015,9,ATLANTIQUE,295000,MAUVAIS,COLINA,270000.0,ANCIEN,EXPERIMENTE,5.0,FAIBLE,0,2,11.76,FAIBLE,2022,9,Thursday,False,ELEVE
3,AUTO-2022-004,2022,8,2022-08-27,PRT-0586-BJ,NIVELEUSE,VOLVO CE,4,BON,BORGOU,VOIRIE URBAINE PARAKOU,ACCROCHAGE MINEUR,RESPONSABLE,MATÉRIEL GRAVE,NON,NSIA BÉNIN,6000000,321000,4851000.0,2,44.0,RÉGLÉ,5,AUTO,NIVELEUSE,VOLVO CE,2020,4,ATLANTIQUE,148000,BON,NSIA BÉNIN,5172000.0,MOYEN,INTERMEDIAIRE,46.0,ELEVE,0,3,34.64,ELEVE,2022,8,Saturday,True,MOYEN
4,AUTO-2022-005,2022,9,2022-09-27,PRT-0395-BJ,CAMION MALAXEUR BÉTON,DAF,2,BON,OUÉMÉ,PISTE RURALE,SORTIE DE ROUTE,RESPONSABILITÉ PARTAGÉE,MATÉRIEL LÉGER,NON,COLINA,500000,243000,210000.0,9,87.0,RÉGLÉ,17,AUTO,CAMION MALAXEUR BÉTON,DAF,2022,2,LITTORAL,80000,BON,COLINA,453000.0,RECENT,EXPERIMENTE,96.0,FAIBLE,0,2,62.09,FAIBLE,2022,9,Tuesday,False,MOYEN


(410, 46)

,id_sinistre,annee,mois,date_declaration,code_chantier,nom_chantier,type_travaux,departement,phase_chantier,nature_sinistre,tiers_implique,engin_implique,assureur,montant_declare_fcfa,franchise_fcfa,montant_indemnise_fcfa,delai_declaration_jours,delai_reglement_jours,statut_dossier,responsabilite,type_sinistre,cout_total_fcfa,niveau_risque,tranche_cout
0,TRC-2022-001,2022,4,2022-04-13,CH-003,ROUTE PARAKOU-N'DALI,TERRASSEMENT,BORGOU,GROS ŒUVRE,CONDUITE EAU ENDOMMAGÉE,SONEB,PELLE HYDRAULIQUE,NSIA BÉNIN,1858000,1016000,0.0,8,0.0,EN EXPERTISE,PORTEO ENTIÈRE,TRC,1016000.0,MOYEN,MOYEN
1,TRC-2022-002,2022,10,2022-10-19,CH-029,ROUTE COVÈ-ZAGNANADO,TERRASSEMENT,ZOU,GROS ŒUVRE,DOMMAGES PROPRIÉTÉ RIVERAINE,RIVERAIN PARTICULIER,CAMION BENNE,UAB,2780000,1208000,0.0,21,0.0,EN COURS,PORTEO ENTIÈRE,TRC,1208000.0,MOYEN,MOYEN
2,TRC-2022-003,2022,8,2022-08-07,CH-007,BITUMAGE NATITINGOU-TANGUIÉTA,BITUMAGE,ATACORA,TERRASSEMENT,DOMMAGES PROPRIÉTÉ RIVERAINE,RIVERAIN PARTICULIER,CHARGEUR,NSIA BÉNIN,4284000,984000,2571000.0,20,156.0,LITIGIEUX,PORTEO PARTIELLE,TRC,3555000.0,MOYEN,MOYEN
3,TRC-2022-004,2022,1,2022-01-17,CH-013,PONT SUR OUÉMÉ À ADJOHOUN,OUVRAGE D'ART,OUÉMÉ,FINITION,EFFONDREMENT DE TALUS,COLLECTIVITÉ LOCALE,CHARGEUR,NSIA BÉNIN,17294000,375000,16101000.0,11,45.0,LITIGIEUX,PORTEO PARTIELLE,TRC,16476000.0,ELEVE,ELEVE
4,TRC-2022-005,2022,10,2022-10-08,CH-014,VOIRIE PARAKOU ZONE INDUSTRIELLE,VRD,BORGOU,TERRASSEMENT,DOMMAGES PROPRIÉTÉ RIVERAINE,RIVERAIN PARTICULIER,NIVELEUSE,UAB,2637000,682000,1562000.0,6,184.0,RÉGLÉ,PORTEO ENTIÈRE,TRC,2244000.0,MOYEN,MOYEN


(166, 24)

In [43]:
# TRUNCATE TABLE FACT
with engine.begin() as conn:

    conn.execute(
        text("""
            TRUNCATE TABLE fact_sinistres CASCADE;
        """)
    )

print("fact_sinistres vidée")

fact_sinistres vidée


In [45]:
# CHARGEMENT DIM_DATE EXISTANTE
dim_date_sql = pd.read_sql(

    "SELECT * FROM dim_date",

    engine

)

display(dim_date_sql.head())

,date_key,date_complete,jour,mois,trimestre,semestre,annee,nom_mois,nom_jour,jour_semaine_num,est_weekend,semaine_annee,est_jour_ferie,est_fin_mois


## 2. Création des dimensions

- dim_date ;
- dim_vehicule ;
- dim_conducteur ;
- dim_chantier ;
- dim_assureur ;
- dim_localisation ;
- dim_type_sinistre.

In [47]:
# CRÉATION DIM_DATE TEMPORAIRE
dim_date = pd.DataFrame()

dim_date["date_complete"] = pd.to_datetime(
    dataset["date_accident"]
)

dim_date["jour"] = dim_date[
    "date_complete"
].dt.day

dim_date["mois"] = dim_date[
    "date_complete"
].dt.month

dim_date["trimestre"] = dim_date[
    "date_complete"
].dt.quarter

dim_date["semestre"] = np.where(

    dim_date["mois"] <= 6,

    1,

    2

)

dim_date["annee"] = dim_date[
    "date_complete"
].dt.year

dim_date["nom_mois"] = dim_date[
    "date_complete"
].dt.month_name()

dim_date["nom_jour"] = dim_date[
    "date_complete"
].dt.day_name()

dim_date["jour_semaine_num"] = dim_date[
    "date_complete"
].dt.weekday

dim_date["est_weekend"] = dim_date[
    "jour_semaine_num"
].isin([5,6])

dim_date = dim_date.drop_duplicates()

In [49]:
# IDENTIFICATION DES NOUVELLES DATES

# Uniformisation des types
dim_date["date_complete"] = pd.to_datetime(
    dim_date["date_complete"]
).dt.date

dim_date_sql["date_complete"] = pd.to_datetime(
    dim_date_sql["date_complete"]
).dt.date

# Suppression doublons internes
dim_date = dim_date.drop_duplicates(
    subset=["date_complete"]
)

# Détection des nouvelles dates
nouvelles_dates = dim_date[
    
    ~dim_date["date_complete"].isin(
        dim_date_sql["date_complete"]
    )
    
]

print("Nombre de nouvelles dates :",
      nouvelles_dates.shape[0])

display(nouvelles_dates.head())

Nombre de nouvelles dates : 343


,date_complete,jour,mois,trimestre,semestre,annee,nom_mois,nom_jour,jour_semaine_num,est_weekend
0,2022-07-17,17,7,3,2,2022,July,Sunday,6,True
1,2022-03-02,2,3,1,1,2022,March,Wednesday,2,False
2,2022-09-15,15,9,3,2,2022,September,Thursday,3,False
3,2022-08-27,27,8,3,2,2022,August,Saturday,5,True
4,2022-09-27,27,9,3,2,2022,September,Tuesday,1,False


In [51]:
# INSERTION DIM_DATE
if len(nouvelles_dates) > 0:

    nouvelles_dates.to_sql(

        "dim_date",

        engine,

        if_exists="append",

        index=False

    )

    print("Nouvelles dates insérées")

else:

    print("Aucune nouvelle date")

Nouvelles dates insérées


In [53]:
dim_vehicule = dataset[

    [
        "immatriculation",
        "type_engin_parc",
        "marque_parc",
        "annee_mise_en_service",
        "age_vehicule_ans_parc",
        "km_parcourus_estimes",
        "etat_general",
        "departement_affectation",
        "classe_age",
        "categorie_km"
    ]

].drop_duplicates()

dim_vehicule.columns = [

    "immatriculation",
    "type_engin",
    "marque",
    "annee_mise_en_service",
    "age_vehicule_ans",
    "km_parcourus_estimes",
    "etat_general",
    "departement_affectation",
    "classe_age",
    "categorie_km"
]

In [59]:
vehicule_sql = pd.read_sql(

    """
    SELECT immatriculation
    FROM dim_vehicule
    """,

    engine

)

In [61]:
nouveaux_vehicules = dim_vehicule[

    ~dim_vehicule["immatriculation"].isin(
        vehicule_sql["immatriculation"]
    )

]

print(nouveaux_vehicules.shape)

(325, 10)


In [63]:
if len(nouveaux_vehicules) > 0:

    nouveaux_vehicules.to_sql(

        "dim_vehicule",

        engine,

        if_exists="append",

        index=False

    )

    print("Nouveaux véhicules insérés")

else:

    print("Aucun nouveau véhicule")

Nouveaux véhicules insérés


In [65]:
dim_conducteur = dataset[

    [
        "conducteur_anciennete_ans",
        "niveau_experience"
    ]

].drop_duplicates()

dim_conducteur["conducteur_id"] = range(

    1,

    len(dim_conducteur) + 1

)

dim_conducteur["tranche_age_conducteur"] = np.where(

    dim_conducteur["conducteur_anciennete_ans"] <= 2,

    "JEUNE",

    "EXPERIMENTE"

)

In [67]:
dim_conducteur.to_sql(

    "dim_conducteur",

    engine,

    if_exists="append",

    index=False

)

print("dim_conducteur alimentée")

dim_conducteur alimentée


In [89]:
# HARMONISATION DES COLONNES

assureur_auto = dataset[["assureur_parc"]].rename(
    columns={"assureur_parc": "assureur"}
)

assureur_trc = trc[["assureur"]]

# CONCATENATION

dim_assureur = pd.concat([
    assureur_auto,
    assureur_trc
]).drop_duplicates()

# RESET INDEX

dim_assureur = dim_assureur.reset_index(drop=True)

# AJOUT ATTRIBUTS

dim_assureur["code_assureur"] = range(
    1,
    len(dim_assureur) + 1
)

dim_assureur["type_contrat"] = "STANDARD"

In [91]:
print(dim_assureur.head())
print(dim_assureur.columns)

     assureur  code_assureur type_contrat
0      COLINA              1     STANDARD
1  SAAR-BÉNIN              2     STANDARD
2  NSIA BÉNIN              3     STANDARD
3       SOBAC              4     STANDARD
4         UAB              5     STANDARD
Index(['assureur', 'code_assureur', 'type_contrat'], dtype='object')


In [93]:
assureur_sql = pd.read_sql(

    """
    SELECT assureur
    FROM dim_assureur
    """,

    engine

)

In [95]:
nouveaux_assureurs = dim_assureur[

    ~dim_assureur["assureur"].isin(
        assureur_sql["assureur"]
    )

]

In [97]:
if len(nouveaux_assureurs) > 0:

    nouveaux_assureurs.to_sql(

        "dim_assureur",

        engine,

        if_exists="append",

        index=False

    )

    print("Assureurs insérés")

else:

    print("Aucun nouvel assureur")

Assureurs insérés


In [99]:
dim_localisation = dataset[

    [
        "departement_accident",
        "axe_routier"
    ]

].drop_duplicates()

dim_localisation.columns = [

    "departement",
    "axe_routier"
]

dim_localisation["nom_departement"] = dim_localisation[
    "departement"
]

dim_localisation["region"] = "BENIN"

dim_localisation["type_zone"] = "URBAINE"

In [101]:
dim_localisation.to_sql(

    "dim_localisation",

    engine,

    if_exists="append",

    index=False

)

print("dim_localisation alimentée")

dim_localisation alimentée


In [103]:
dim_type_sinistre = dataset[

    [
        "type_sinistre",
        "type_accident",
        "responsabilite",
        "tiers_implique",
        "gravite",
        "statut_dossier"
    ]

].drop_duplicates()

dim_type_sinistre["nature_sinistre"] = "AUTO"

## 3. Création de fact_sinistres

In [115]:
dim_type_sinistre["tiers_implique"] = (
    dim_type_sinistre["tiers_implique"]
    .map({
        "OUI": True,
        "NON": False
    })
)

In [117]:
dim_type_sinistre.to_sql(

    "dim_type_sinistre",

    engine,

    if_exists="append",

    index=False

)

print("dim_type_sinistre alimentée")

dim_type_sinistre alimentée


In [119]:
date_sql = pd.read_sql(
    "SELECT * FROM dim_date",
    engine
)

vehicule_sql = pd.read_sql(
    "SELECT * FROM dim_vehicule",
    engine
)

conducteur_sql = pd.read_sql(
    "SELECT * FROM dim_conducteur",
    engine
)

assureur_sql = pd.read_sql(
    "SELECT * FROM dim_assureur",
    engine
)

localisation_sql = pd.read_sql(
    "SELECT * FROM dim_localisation",
    engine
)

type_sinistre_sql = pd.read_sql(
    "SELECT * FROM dim_type_sinistre",
    engine
)

In [121]:
fact = dataset.copy()

fact["date_complete"] = pd.to_datetime(
    fact["date_accident"]
)

In [133]:
fact["date_complete"] = pd.to_datetime(
    fact["date_complete"]
)

date_sql["date_complete"] = pd.to_datetime(
    date_sql["date_complete"]
)

In [135]:
fact = fact.merge(

    date_sql[
        [
            "date_key",
            "date_complete"
        ]
    ],

    on="date_complete",

    how="left"

)

In [139]:
fact = fact.merge(

    vehicule_sql[
        [
            "vehicule_key",
            "immatriculation"
        ]
    ],

    on="immatriculation",

    how="left"

)

In [141]:
fact = fact.merge(

    conducteur_sql[
        [
            "conducteur_key",
            "conducteur_anciennete_ans"
        ]
    ],

    on="conducteur_anciennete_ans",

    how="left"

)

In [160]:
print(fact.columns)

Index(['id_sinistre', 'annee', 'mois', 'date_accident', 'immatriculation',
       'type_engin_sinistre', 'marque_sinistre', 'age_vehicule_ans_sinistre',
       'etat_vehicule', 'departement_accident', 'axe_routier', 'type_accident',
       'responsabilite', 'gravite', 'tiers_implique', 'assureur_sinistre',
       'montant_declare_fcfa', 'franchise_fcfa', 'montant_indemnise_fcfa',
       'delai_declaration_jours', 'delai_reglement_jours', 'statut_dossier',
       'conducteur_anciennete_ans', 'type_sinistre', 'type_engin_parc',
       'marque_parc', 'annee_mise_en_service', 'age_vehicule_ans_parc',
       'departement_affectation', 'km_parcourus_estimes', 'etat_general',
       'assureur_parc', 'cout_total_fcfa', 'classe_age', 'niveau_experience',
       'score_delai', 'niveau_risque', 'sinistre_grave', 'frequence_sinistre',
       'score_risque', 'tranche_cout', 'annee_accident', 'mois_accident',
       'jour_semaine', 'est_weekend', 'categorie_km', 'date_complete',
       'conducteur_k

In [162]:
print(assureur_sql.columns)

Index(['assureur_key', 'assureur', 'code_assureur', 'type_contrat'], dtype='object')


In [164]:
fact = fact.merge(

    assureur_sql[
        [
            "assureur_key",
            "assureur"
        ]
    ],

    left_on="assureur_parc",

    right_on="assureur",

    how="left"

)

In [166]:
fact = fact.drop(
    columns=["assureur"]
)

In [168]:
print(
    fact[
        [
            "assureur_parc",
            "assureur_key"
        ]
    ].head()
)

  assureur_parc  assureur_key
0        COLINA             1
1        COLINA             1
2        COLINA             1
3        COLINA             1
4        COLINA             1


In [170]:
fact = fact.merge(

    localisation_sql[
        [
            "localisation_key",
            "departement",
            "axe_routier"
        ]
    ],

    left_on=[
        "departement_accident",
        "axe_routier"
    ],

    right_on=[
        "departement",
        "axe_routier"
    ],

    how="left"

)

In [172]:
fact = fact.merge(

    type_sinistre_sql[
        [
            "type_sinistre_key",
            "gravite"
        ]
    ],

    on="gravite",

    how="left"

)

## 4. Création de fact_predictions

## 5. Injection des données

- to_sql ;
- insertions.

In [177]:
fact.columns

Index(['id_sinistre', 'annee', 'mois', 'date_accident', 'immatriculation',
       'type_engin_sinistre', 'marque_sinistre', 'age_vehicule_ans_sinistre',
       'etat_vehicule', 'departement_accident', 'axe_routier', 'type_accident',
       'responsabilite', 'gravite', 'tiers_implique', 'assureur_sinistre',
       'montant_declare_fcfa', 'franchise_fcfa', 'montant_indemnise_fcfa',
       'delai_declaration_jours', 'delai_reglement_jours', 'statut_dossier',
       'conducteur_anciennete_ans', 'type_sinistre', 'type_engin_parc',
       'marque_parc', 'annee_mise_en_service', 'age_vehicule_ans_parc',
       'departement_affectation', 'km_parcourus_estimes', 'etat_general',
       'assureur_parc', 'cout_total_fcfa', 'classe_age', 'niveau_experience',
       'score_delai', 'niveau_risque', 'sinistre_grave', 'frequence_sinistre',
       'score_risque', 'tranche_cout', 'annee_accident', 'mois_accident',
       'jour_semaine', 'est_weekend', 'categorie_km', 'date_complete',
       'conducteur_k

In [179]:
fact = fact.rename(
    columns={
        "conducteur_key_y": "conducteur_key",
        "localisation_key_y": "localisation_key",
        "type_sinistre_key_y": "type_sinistre_key"
    }
)

In [181]:
fact = fact.drop(
    columns=[
        "conducteur_key_x",
        "localisation_key_x",
        "type_sinistre_key_x",
        "departement_x",
        "departement_y"
    ],
    errors="ignore"
)

In [183]:
fact_final = fact[

    [

        "date_key",
        "vehicule_key",
        "conducteur_key",
        "assureur_key",
        "localisation_key",
        "type_sinistre_key",

        "montant_declare_fcfa",
        "franchise_fcfa",
        "montant_indemnise_fcfa",
        "cout_total_fcfa",

        "delai_declaration_jours",
        "delai_reglement_jours",

        "score_risque"

    ]

]

In [ ]:
fact_final.to_sql(

    "fact_sinistres",

    engine,

    if_exists="append",

    index=False

)

print("fact_sinistres alimentée")

In [ ]:
test_fact = pd.read_sql(

    """
    SELECT *
    FROM fact_sinistres
    LIMIT 10
    """,

    engine

)

display(test_fact)

In [155]:
print("Fact rows :", fact_final.shape[0])

print("Vehicules :", len(vehicule_sql))

print("Assureurs :", len(assureur_sql))

print("Localisations :", len(localisation_sql))

NameError: name 'fact_final' is not defined

In [157]:
rapport = pd.DataFrame({

    "Table": [

        "dim_date",
        "dim_vehicule",
        "dim_conducteur",
        "dim_assureur",
        "dim_localisation",
        "dim_type_sinistre",
        "fact_sinistres"

    ],

    "Nb_lignes": [

        len(date_sql),
        len(vehicule_sql),
        len(conducteur_sql),
        len(assureur_sql),
        len(localisation_sql),
        len(type_sinistre_sql),
        len(fact_final)

    ]

})

rapport.to_csv(

    "../reports/tables/rapport_alimentation_dwh.csv",

    index=False

)

print("Rapport exporté")

NameError: name 'fact_final' is not defined

## Synthèse de l’alimentation du DWH

### Opérations réalisées
- Connexion PostgreSQL
- Chargement dimensions
- Chargement table de faits
- Gestion des doublons
- Création des clés analytiques

### Résultat
Le Data Warehouse est maintenant prêt pour :
- Power BI
- KPI décisionnels
- Reporting
- Analyse OLAP
- Machine Learning avancé
- Système intelligent d’aide à la décision